# Business Simulation — Credit Risk Analytics

**Presenter:** Senior Data Scientist, Horizon Lending Inc.  
**Audience:** Bank Management Team (CRO, CFO, CEO, Head of Underwriting)  
**Date:** July 2026  
**Model:** XGBoost Classifier (AUC 0.81, selected in `03_modeling.ipynb`)  

---

## Executive Summary

This presentation translates the machine learning model into **dollar-denominated business outcomes**. We compare three lending policies and show how the AI model improves profitability, reduces losses, and enables more precise risk segmentation.

**The question this analysis answers:**  
*"If we deploy this model, how much more money do we make compared to our current underwriting approach?"*

---

## Assumptions

All dollar figures are based on a simulated portfolio of **50,000 loan applicants** consistent with Horizon Lending's current volume.

| Parameter | Value | Source / Rationale |
|---|---|---|
| Number of applicants | 50,000 | Horizon's monthly application volume |
| Average loan amount | $12,500 | Portfolio average across all products |
| Average loan term | 36 months | Most common product term |
| Average interest rate (APR) | 15.0% | Horizon's near-prime market rate |
| Revenue per performing loan | $3,042 | Total interest over 36 months at 15% APR |
| Loss Given Default (LGD) | 60% | Industry standard for unsecured personal loans; 40% recovery rate |
| Cost of funds | 4.0% | Horizon's weighted average cost of capital |
| Per-loan origination cost | $150 | Underwriting, processing, documentation |
| Collections cost per delinquent account | $75 | Average cost to pursue a defaulted account |

**Net revenue per performing loan:**
- Total interest received: $3,042
- Cost of funds (4% over 36 months): $750
- Origination cost: $150
- **Net revenue if loan performs: $2,142**

**Net loss per defaulted loan:**
- Principal loss (60% LGD × $12,500): $7,500
- Uncollected interest: $3,042
- Collections cost: $75
- **Net loss if loan defaults: $10,617**

---

## Simulation 1: Approve Everyone

**Policy:** Accept every applicant regardless of risk profile. No underwriting filter.

**Why this scenario:** Serves as the worst-case reference point. Demonstrates why some form of underwriting is essential.

| Metric | Value | Calculation |
|---|---|---|
| Applications received | 50,000 | — |
| **Approval rate** | **100%** | All 50,000 approved |
| Loans funded | 50,000 | — |
| Total principal lent | $625,000,000 | 50,000 × $12,500 |
| Expected default rate | 8.0% | Historical average for near-prime applicants |
| Defaulted loans | 4,000 | 8% of 50,000 |
| Performing loans | 46,000 | 92% of 50,000 |
| **Total revenue from performing loans** | **$98,532,000** | 46,000 × $2,142 |
| **Total loss from defaults** | **−$42,468,000** | 4,000 × $10,617 |
| **Net portfolio profit** | **$56,064,000** | Revenue − Loss |
| **Expected loss rate** | **6.79%** | $42.47M / $625M principal |
| **Risk-adjusted return** | **8.97%** | $56.06M / $625M principal |

**Insight:** Approving everyone generates $56M in profit — but at enormous risk. A 1 percentage-point increase in the default rate would erase $6.2M of profit. There is no risk control.

---

## Simulation 2: Current Policy (Rules-Based)

**Policy:** Horizon's current rule-based underwriting.

- Decline if `EXT_SOURCE_2 < 0.30` (bottom quartile of external credit scores)
- Decline if `DTI > 0.45` (debt payments exceed 45% of income)
- Decline if `PREV_DEFAULT_RATE > 0` (any past default on a Horizon loan)
- Decline if `AMT_CREDIT > 8 × AMT_INCOME_TOTAL` (LTI ratio exceeds 8)

These rules are applied sequentially. An applicant is declined if they fail any single rule.

| Rule | Applicants Filtered | Remaining | Default Rate of Filtered | Default Rate of Remaining |
|---|---|---|---|---|
| Start | — | 50,000 | — | 8.0% |
| EXT_SOURCE_2 < 0.30 | 5,500 declined | 44,500 | 18.5% | 6.7% |
| DTI > 0.45 | 3,200 declined | 41,300 | 15.2% | 5.5% |
| PREV_DEFAULT_RATE > 0 | 1,800 declined | 39,500 | 22.0% | 4.2% |
| LTI > 8 | 1,200 declined | 38,300 | 12.0% | 3.7% |

| Metric | Value |
|---|---|
| Applications received | 50,000 |
| Applications declined (rules) | 11,700 |
| **Approval rate** | **76.6%** |
| Loans funded | 38,300 |
| Total principal lent | $478,750,000 |
| Expected default rate (approved pool) | 3.7% |
| Defaulted loans | 1,417 |
| Performing loans | 36,883 |
| **Total revenue from performing loans** | **$79,002,000** |
| **Total loss from defaults** | **−$15,044,000** |
| **Net portfolio profit** | **$63,958,000** |
| **Expected loss rate** | **3.14%** |
| **Risk-adjusted return** | **13.36%** |

**Insight:** The current rules-based policy improves risk-adjusted return from 8.97% to 13.36% — a significant improvement over approving everyone. However, the rules are blunt: they decline 11,700 applicants (23.4%) but still miss 1,417 defaults. Many of the declined applicants might have been profitable.

**Key limitation:** The rules are independent. They do not capture interactions (e.g., a young applicant with high DTI but excellent external score might be safe, but gets declined by the DTI rule alone).

---

## Simulation 3: AI-Assisted Policy (Model-Based)

**Policy:** Use the XGBoost model's predicted default probability (PD) with an optimised decision threshold. Approve if `PD < threshold`; decline if `PD >= threshold`.

The optimal threshold was determined in the evaluation stage by maximising portfolio profit (F2-optimal threshold: **0.18**).

### 3A: AI Model at Optimal Threshold (PD < 0.18 → Approve)

| Metric | Value |
|---|---|
| Applications received | 50,000 |
| Applications declined (PD >= 0.18) | 9,500 |
| **Approval rate** | **81.0%** |
| Loans funded | 40,500 |
| Total principal lent | $506,250,000 |
| Expected default rate (approved pool) | 2.8% |
| Defaulted loans | 1,134 |
| Performing loans | 39,366 |
| **Total revenue from performing loans** | **$84,322,000** |
| **Total loss from defaults** | **−$12,040,000** |
| **Net portfolio profit** | **$72,282,000** |
| **Expected loss rate** | **2.38%** |
| **Risk-adjusted return** | **14.28%** |

### 3B: AI Model — Conservative Threshold (PD < 0.10 → Approve)

Tighten the threshold to approve only the safest applicants.

| Metric | Value |
|---|---|
| **Approval rate** | **68.0%** |
| Loans funded | 34,000 |
| Total principal lent | $425,000,000 |
| Expected default rate | 1.5% |
| Defaulted loans | 510 |
| Performing loans | 33,490 |
| **Total revenue from performing loans** | **$71,735,000** |
| **Total loss from defaults** | **−$5,415,000** |
| **Net portfolio profit** | **$66,320,000** |
| **Expected loss rate** | **1.27%** |
| **Risk-adjusted return** | **15.60%** |

### 3C: AI Model — Aggressive Threshold (PD < 0.30 → Approve)

Loosen the threshold to approve more applicants, accepting higher defaults.

| Metric | Value |
|---|---|
| **Approval rate** | **90.0%** |
| Loans funded | 45,000 |
| Total principal lent | $562,500,000 |
| Expected default rate | 5.0% |
| Defaulted loans | 2,250 |
| Performing loans | 42,750 |
| **Total revenue from performing loans** | **$91,570,000** |
| **Total loss from defaults** | **−$23,888,000** |
| **Net portfolio profit** | **$67,682,000** |
| **Expected loss rate** | **4.25%** |
| **Risk-adjusted return** | **12.03%** |

---

## Scenario Comparison: Three Policies Side by Side

| Metric | Approve Everyone | Current Rules | AI (Optimal Threshold) |
|---|---|---|---|
| Approval rate | 100.0% | 76.6% | 81.0% |
| Loans funded | 50,000 | 38,300 | 40,500 |
| Principal lent | $625.0M | $478.8M | $506.3M |
| Default rate (approved) | 8.0% | 3.7% | 2.8% |
| Defaulted loans | 4,000 | 1,417 | 1,134 |
| Total loss | −$42.5M | −$15.0M | −$12.0M |
| Total revenue | $98.5M | $79.0M | $84.3M |
| **Net profit** | **$56.1M** | **$64.0M** | **$72.3M** |
| Loss rate (% of principal) | 6.79% | 3.14% | 2.38% |
| Risk-adjusted return | 8.97% | 13.36% | **14.28%** |

**Key takeaways for management:**

1. **Higher profit:** The AI policy generates **$72.3M** vs. **$64.0M** under current rules — a **$8.3M (13%) improvement**.
2. **More loans, fewer defaults:** The AI policy approves **2,200 more loans** than current rules (81.0% vs. 76.6%) *and* has a **lower default rate** (2.8% vs. 3.7%). This is the win-win: the model identifies safe borrowers that the blunt rules miss.
3. **Better risk-adjusted return:** 14.28% vs. 13.36% — the AI model earns more profit per dollar lent.

---

## Threshold Sensitivity Analysis

The decision threshold is the most important business lever. Changing it directly controls the trade-off between approval volume and default risk.

| Threshold (PD cutoff) | Approval Rate | Default Rate | Net Profit | Loss Rate | Risk-Adj. Return |
|---|---|---|---|---|---|
| 0.05 | 55% | 0.8% | $58.5M | 0.7% | 17.1% |
| 0.10 | 68% | 1.5% | $66.3M | 1.3% | 15.6% |
| **0.18 (optimal)** | **81%** | **2.8%** | **$72.3M** | **2.4%** | **14.3%** |
| 0.25 | 87% | 4.2% | $70.8M | 3.6% | 12.9% |
| 0.30 | 90% | 5.0% | $67.7M | 4.3% | 12.0% |
| 0.40 | 94% | 6.3% | $61.5M | 5.4% | 10.4% |
| 0.50 | 96% | 7.2% | $58.2M | 6.1% | 9.7% |

**What this tells us:**
- The profit curve is **inverted-U shaped**. At very low thresholds (too strict), we decline too many good borrowers and leave money on the table. At very high thresholds (too loose), defaults erode profits.
- The optimal threshold (0.18) is the **sweet spot** that maximizes net profit.
- For every $1M of profit the bank wants to target, there is a corresponding threshold and risk level. This enables strategic decision-making: "If we want to grow, we accept a lower return but higher volume."

---

## Recommended Visualizations for Management Presentation

### Chart 1: Three-Policy Comparison Bar Chart

```
Net Profit by Policy ($M)
┌──────────────────────────────────────────────────────┐
│ Approve Everyone     ██████████████████     $56.1M   │
│ Current Rules        ██████████████████████  $64.0M  │
│ AI Model (Optimal)   █████████████████████████ $72.3M│
└──────────────────────────────────────────────────────┘
       $0    $10    $20    $30    $40    $50    $60    $70
```

**What it shows:** The AI model delivers $8.3M more profit than current rules and $16.2M more than approving everyone.

---

### Chart 2: Profit Curve vs. Threshold (Line Chart)

```
                           ┌──┐
                         ┌─┘  │  Optimal Threshold (0.18)
                        ┌┘    │  → $72.3M profit
                       ┌┘     │
              ┌────────┘      └──┐
       ┌──────┘                    └──────┐
  ┌────┘                                    └────┐
  │                                                  │
  0.00  0.05  0.10  0.15  0.20  0.25  0.30  0.35  0.40
          Decision Threshold (PD Cutoff)
```

**What it shows:** The inverted-U shape. Two thresholds can yield the same profit: the steeper left side (too strict, leaving money on the table) and the gentler right side (too loose, defaults eating into profits). The peak is the optimal balance.

**Management action:** Use the slider on this chart in an interactive dashboard to explore. "What if we set the threshold at 0.25?" → Shows $70.8M profit with 87% approval.

---

### Chart 3: Approval Rate vs. Default Rate (Scatter / Trade-Off Plot)

```
Default Rate
   10% │                                           ⋆ (Approve All)
       │
    8% │
       │                              ⋆ (Threshold 0.40)
    6% │
       │                    ⋆ (Threshold 0.25)
    4% │          ⋆ (Current Rules)   
       │     ⋆ (AI Optimal 0.18)
    2% │  ⋆ (Conservative 0.10)
       │⋆ (Very Conservative 0.05)
    0% └────────────────────────────────────────────
       50%    60%    70%    80%    90%    100%
                    Approval Rate
```

**What it shows:** The AI policy (optimal) achieves both a *higher* approval rate AND a *lower* default rate than the current rules policy. This is the "efficient frontier" of lending — the AI model is strictly better.

---

### Chart 4: Portfolio Composition by Risk Bucket (Stacked Bar)

```
Portfolio Distribution by Risk Bucket

Approve All   ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
              (8% default across all)

Current Rules ████████████████████████████░░░░░░░░░░░████████
              A (safe)              B       C      D   E (risky)

AI Optimal    ██████████████████████████████████████░░░░░░░░
              A (safe)              B       C      D  E

Legend: A = PD < 0.05 (very safe), B = 0.05–0.10, C = 0.10–0.18, D = 0.18–0.30, E > 0.30
```

**What it shows:** The AI model shifts more applicants into the safe buckets (A, B, C) compared to current rules, while declining more of the high-risk buckets (D, E). The model is more precise.

---

## Collections Simulation: Early Intervention

Beyond approval decisions, the model also improves **collections efficiency**. By ranking all loans by PD score at origination, the collections team knows exactly which accounts to monitor.

**Simulation:** If the collections team intervenes on the top 5% highest-risk loans (as ranked by the model), and intervention reduces LGD by 20%:

| Metric | Without AI Collections | With AI Collections |
|---|---|---|
| Number of defaults (AI policy) | 1,134 | 1,134 |
| Defaults caught by top-5% flag | — | 454 (40% of all defaults) |
| LGD on intervened accounts | 60% | 48% (20% reduction) |
| Savings per intervened default | — | $1,500 (12% of $12,500) |
| Total loss savings | — | $681,000 |
| Collections cost (2,025 accounts × $75) | — | $151,875 |
| Net collections benefit | — | **$529,125** |

**Insight:** The model pays for itself through collections efficiency alone. The early-warning system reduces default severity on the accounts most likely to default.

---

## Recommendations for Management

| # | Recommendation | Expected Impact | Priority |
|---|---|---|---|
| 1 | **Deploy the AI model at PD threshold 0.18** | +$8.3M annual profit vs. current rules | Immediate (Q3) |
| 2 | **Implement collections scoring** using model PD ranks | +$0.5M annual loss reduction | Immediate (Q3) |
| 3 | **Set up monitoring dashboard** for threshold performance | Ensures profit target is maintained | Q4 |
| 4 | **Run quarterly threshold reviews** to adjust for economic changes | Prevents profit erosion from concept drift | Ongoing |
| 5 | **Phase out rule-based underwriting** in favour of model-only | Reduces operational complexity, eliminates contradictory rules | Q1 next year |

### Risk Considerations

- The model is a **decision aid**, not a replacement for human judgment. Edge cases (e.g., self-employed with limited bureau data) should be reviewed manually.
- Regulatory approval may be required before fully automated deployment. Run the model in parallel with current rules for one quarter to validate real-world performance.
- Monitor for fairness: the approval rate should not differ materially across demographic groups beyond what is justified by risk.

---

## Appendix: Simulation Methodology

### Profit Calculation Formula

For each loan `i` with probability of default `PD_i`, loan amount `A_i`, and interest rate `r`:

```
Expected Profit_i = (1 - PD_i) × Net_Revenue_i - PD_i × Net_Loss_i

Where:
  Net_Revenue_i = Total_Interest_i - Cost_of_Funds_i - Origination_Cost
  Net_Loss_i    = Principal_Loss_i + Uncollected_Interest_i + Collections_Cost

Portfolio_Profit = sum(Expected_Profit_i) for all approved loans
```

### Policy Simulation Steps

1. Generate PD predictions for all 50,000 applicants using the trained XGBoost model.
2. Apply the decision threshold: approve if `PD_i < threshold`, decline otherwise.
3. For each approved loan, compute expected profit using the formula above.
4. Aggregate across all approved loans to get portfolio-level metrics.
5. Repeat for each threshold value (0.01 to 0.50 in steps of 0.005).
6. Find the threshold that maximizes total portfolio profit.

### Scenario Comparisons

- **Approve everyone:** Approve all 50,000 applicants regardless of PD. Baseline worst case.
- **Current rules:** Apply rule-based filters sequentially. Count how many pass each rule. Compute the default rate and profit for the approved pool.
- **AI model:** Apply PD threshold. Compute metrics for the approved pool.

---

*End of Business Simulation. Ready for management presentation.*